# **Маска как обучаемый параметр: Meaningful perturbation, Extremal perturbations, Real-time saliency и L2X**

Практика к модулю [«Объяснение через вмешательство»](https://ai-interpretability.school), урок «Обучаемые baselines».

В практике `PRACTICE_perturbation_rise` маски перебирались: блок за блоком у абляции, окно за окном у occlusion, случайная маска за случайной у RISE. В уроке «Обучаемые baselines» маску не перебирают, а ищут: маска — тензор, целевая функция — две строки, дальше обычный оптимизатор. Здесь мы проверим это на тех же модели и снимке, что в `PRACTICE_perturbation_rise` (ResNet-50 и фотография свиньи), а потом перенесем оптимизацию из времени объяснения во время обучения на MNIST.

1. Meaningful perturbation руками: игра на удаление, маска на `torch.optim.Adam`, и что будет без регуляризации.
2. Extremal perturbations библиотекой `torchray`: маски заданной площади 5, 10 и 20%.
3. Сравнение с occlusion из соседней практики по deletion и insertion.
4. Real-time saliency на MNIST: сеть-маскировщик, которая выдает маску за один проход, и сколько стоит одно объяснение.
5. L2X на MNIST: выбор $k$ блоков через Concrete-распределение и проверка, чье поведение описывает выбранное подмножество.

Оптимизация маски на ResNet-50 — сотни прямых и обратных проходов на одно объяснение. Включите в Colab среду с GPU («Среда выполнения → Сменить среду выполнения → T4 GPU»): на CPU разделы 1–3 идут в несколько раз дольше.

Автор фото поросенка: [Brett Sayles, Pexels](https://www.pexels.com/photo/4966539/), лицензия Pexels.

In [ ]:
!pip install -q torchray==1.0.0.2

In [ ]:
import time
import warnings
from io import BytesIO

import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from torchvision import datasets, models, transforms
from torchvision.models import ResNet50_Weights
from torchray.attribution.extremal_perturbation import extremal_perturbation, simple_reward

warnings.filterwarnings('ignore', message='torch.meshgrid')   # предупреждение изнутри torchray
torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad_(False)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
RAW = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/'

def load(name):
    return Image.open(BytesIO(requests.get(RAW + name).content)).convert('RGB')

image = load('piglet.jpg').resize((224, 224))
x = transform(image).unsqueeze(0).to(DEVICE)
blur = transform(image.filter(ImageFilter.GaussianBlur(12))).unsqueeze(0).to(DEVICE)   # baseline из PRACTICE_perturbation_rise
categories = [s.strip() for s in requests.get(RAW + 'imagenet_classes.txt').text.splitlines()]

def predict(batch):
    with torch.no_grad():
        return torch.cat([model(batch[s:s + 32]).softmax(1)[:, target] for s in range(0, len(batch), 32)])

with torch.no_grad():
    target = model(x).argmax().item()
base_score = predict(x).item()
print(f'устройство: {DEVICE}')
print(f'класс: {target} — {categories[target]}, уверенность {base_score:.3f}')

## 1. Meaningful perturbation: игра на удаление

Постановка из урока ([Fong и Vedaldi, 2017](https://arxiv.org/abs/1704.03296)):

$$m^{\ast}=\arg\min_{m\in[0,1]^{\Lambda}}\; f_c(\Phi(x,m))+\lambda\,\mathrm{mean}(1-m),\qquad \Phi(x,m)=x\odot m+b\odot(1-m).$$

Здесь $m$ — маска, единица в позиции значит «оставить», ноль — «закрыть»; $b$ — размытый снимок, тот же baseline, что в `PRACTICE_perturbation_rise`; $f_c$ — уверенность ResNet-50 в классе hog; $\lambda$ — цена закрытия единицы площади. Штраф за площадь записан через среднее, а не через сумму: так $\lambda$ не зависит от размера картинки.

Чем это отличается от статьи. У Fong и Vedaldi порча «blur» — размытие с ядром, ширина которого в каждой точке зависит от маски (наибольшее стандартное отклонение $\sigma_0=10$), штрафы записаны суммой с $\lambda_1=10^{-4}$ и $\lambda_2=10^{-2}$, маска после растяжения еще размывается ядром $\sigma_m=5$, а начальная маска — наименьший круг, который гасит уверенность на 99%. Здесь порча — смесь с одним размытым снимком `blur`: тот же baseline, что у RISE, occlusion и метрик раздела 3, иначе их нельзя сравнивать. Остальные отличия метода не меняют: запись штрафов средним меняет только масштаб $\lambda$, а маска из единиц на старте и дрожание испорченного входа вместо дрожания маски — детали оптимизации.

Функция `deletion_game` — это весь метод. Маска инициализируется единицами, веса модели заморожены, шаг Adam меняет только маску. Три защиты от состязательных узоров из шага «Состязательность и маски» включаются аргументами:

- `coarse` — маска на сетке `coarse` $\times$ `coarse`, растянутая билинейной интерполяцией до $224\times224$; `None` — маска в полном разрешении;
- `tv` — вес штрафа на полную вариацию (у Fong и Vedaldi степень перепада $\beta=3$);
- `jitter` — случайный сдвиг испорченного входа на каждом шаге до 4 пикселей.

In [ ]:
def upsample(m, coarse):
    return m if coarse is None else F.interpolate(m, size=(224, 224), mode='bilinear', align_corners=False)

def deletion_game(lam, coarse=28, tv=2.0, jitter=True, steps=300, seed=0):
    """argmin_m f_c(Phi(x, m)) + lam * mean(1 - m) + tv * TV(m), Phi — смесь со снимком blur."""
    torch.manual_seed(seed)
    side = 224 if coarse is None else coarse
    m = torch.ones(1, 1, side, side, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([m], lr=0.1)
    for _ in range(steps):
        up = upsample(m, coarse)
        mixed = x * up + blur * (1 - up)
        if jitter:
            dy, dx = [int(v) for v in torch.randint(-4, 5, (2,))]
            mixed = torch.roll(mixed, shifts=(dy, dx), dims=(2, 3))
        loss = model(mixed).softmax(1)[0, target] + lam * (1 - up).mean()
        if tv:
            loss = loss + tv * (((up[..., 1:, :] - up[..., :-1, :]).abs() ** 3).mean()
                                + ((up[..., :, 1:] - up[..., :, :-1]).abs() ** 3).mean())
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():
            m.clamp_(0, 1)
    with torch.no_grad():
        up = upsample(m, coarse)
        score = predict(x * up + blur * (1 - up)).item()
    return up[0, 0].detach(), score, (1 - up).mean().item()

t0 = time.time()
mask_raw, f_raw, closed_raw = deletion_game(0.5, coarse=None, tv=0.0, jitter=False)
mask_reg, f_reg, closed_reg = deletion_game(0.5)
print(f'два запуска по 300 шагов: {time.time() - t0:.0f} с на {DEVICE}')
print(f'{"маска":44}{"закрыто площади":>18}{"уверенность после порчи":>26}')
print(f'{"полное разрешение, без штрафов":44}{closed_raw:>18.1%}{f_raw:>26.3f}')
print(f'{"сетка 28x28, полная вариация, дрожание":44}{closed_reg:>18.1%}{f_reg:>26.3f}')

In [ ]:
def show_mask(ax, closed, title):
    ax.imshow(image)
    ax.imshow(closed.cpu().numpy(), cmap='magma', alpha=0.6, vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image); axes[0].set_title(f'снимок, уверенность {base_score:.2f}', fontsize=10); axes[0].axis('off')
show_mask(axes[1], 1 - mask_raw, f'без регуляризации: закрыто {closed_raw:.1%}, f = {f_raw:.2f}')
show_mask(axes[2], 1 - mask_reg, f'с регуляризацией: закрыто {closed_reg:.1%}, f = {f_reg:.2f}')
plt.tight_layout()
plt.show()

Яркое на картинках — закрытое, $1-m$. Обе маски решают задачу оптимизации: уверенность в классе hog падает почти до нуля. Маска без регуляризации делает это россыпью отдельных пикселей на крохотной доле площади — это состязательный узор, а не объяснение: такие пиксели не несут ничего, что человек назвал бы свиньей. Маска на грубой сетке со штрафом на полную вариацию и дрожанием закрывает связные пятна на голове животного. Регуляризация здесь — часть определения метода: без нее метод отвечает на вопрос о хрупкости сети, а не о том, где на снимке класс.

**Задание 1.** Постройте семейство масок по $\lambda$, как в шаге «Пример» урока: запустите `deletion_game` с регуляризацией при $\lambda\in\{20,\,8,\,1,\,0.1\}$ и для каждого значения выведите долю закрытой площади и уверенность после порчи. При каком $\lambda$ маска пуста? Вложены ли маски друг в друга, как в игрушечном примере урока? Проверьте вложенность долей пикселей с $1-m>0.5$, которые есть у меньшего $\lambda$, но нет у большего.

In [ ]:
# Ваш код здесь

## 2. Extremal perturbations: площадь задают заранее

У Extremal perturbations ([Fong, Patrick и Vedaldi, 2019](https://arxiv.org/abs/1910.08485)) игра обратная — на сохранение: найти маску заданной площади $a$, под которой класс сохраняется лучше всего, а все вне маски размыто. Гладкость встроена в семейство масок (маска строится из грубой сетки с шагом 7 и сглаживанием), а площадь удерживается штрафом на отсортированные значения маски. Все это внутри `torchray.attribution.extremal_perturbation`. Аргументы:

- `areas` — список долей площади, по маске на каждую; возьмем 5, 10 и 20%;
- `reward_func=simple_reward` — целевая функция статьи: оценка класса до softmax; это и умолчание библиотеки;
- `max_iter=800` — число шагов оптимизации, умолчание библиотеки. В статье шагов SGD с моментом 1600, а вес штрафа на площадь удваивают на трети и двух третях пути; в `torchray` по умолчанию шагов 800, и вес штрафа растет плавно, в 1.0035 раза за шаг.

Сохраненную уверенность посчитаем тем же способом, что в разделе 1: под маской снимок, вне маски — размытый снимок `blur`.

In [ ]:
AREAS = [0.05, 0.1, 0.2]
torch.manual_seed(0)
t0 = time.time()
ep_masks, _ = extremal_perturbation(model, x, target, areas=AREAS, reward_func=simple_reward, max_iter=800)
ep_masks = ep_masks[:, 0].detach()
print(f'три маски за {time.time() - t0:.0f} с на {DEVICE}')
print(f'{"заданная площадь":>18}{"среднее маски":>16}{"доля m > 0.5":>16}{"уверенность под маской":>26}')
for a, m in zip(AREAS, ep_masks):
    kept = predict(x * m + blur * (1 - m)).item()
    print(f'{a:>18.0%}{m.mean().item():>16.3f}{(m > 0.5).float().mean().item():>16.3f}{kept:>26.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, a, m in zip(axes, AREAS, ep_masks):
    ax.imshow(np.asarray(image) / 255 * (0.2 + 0.8 * m.cpu().numpy()[..., None]))
    ax.set_title(f'площадь {a:.0%}', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

Яркое здесь — то, что маска оставила. Вместо одной карты получается семейство, и читают его по таблице. Среднее маски показывает, насколько точно штраф удержал заданную площадь, доля пикселей с $m>0.5$ — насколько маска бинарна, а последний столбец — хватает ли этой площади классу. На этом снимке уверенность под маской растет с площадью: 0.115 при 5%, 0.254 при 10% и 0.359 при 20%, меньше половины исходной 0.926. Класс hog возвращается постепенно, а площадь, на которой он сохраняется целиком, в этот перебор не попала. Так и предлагают читать семейство Fong, Patrick и Vedaldi.

**Задание 2.** Сравните ответы двух игр. Возьмите маску Extremal perturbations на площади 10% (пиксели с $m>0.5$) и те 10% пикселей, которые сильнее всего закрыла маска Meaningful perturbation с регуляризацией из раздела 1 (`1 - mask_reg`). Посчитайте IoU двух множеств и корреляцию Пирсона карт `ep_masks[1]` и `1 - mask_reg`. Совпадают ли ответы игры на сохранение и игры на удаление?

In [ ]:
# Ваш код здесь

## 3. Сравнение с occlusion по deletion и insertion

Метрики те же, что в разделе 5 `PRACTICE_perturbation_rise`: deletion убирает пиксели по убыванию важности и смотрит, как падает уверенность (ниже площадь — лучше), insertion возвращает пиксели в размытый снимок (выше — лучше). Карта важности у Meaningful perturbation — закрытость $1-m$, у Extremal perturbations — сумма трех масок, как в статье: пиксель, который оставлен уже на 5% площади, важнее пикселя, который появляется только на 20%. Occlusion — окно 64 с шагом 16, как в соседней практике, и случайная карта для проверки метрик.

In [ ]:
def occlusion(size, stride):
    cells = [(i, j) for i in range(0, 224 - size + 1, stride) for j in range(0, 224 - size + 1, stride)]
    batch = x.repeat(len(cells), 1, 1, 1)
    for k, (i, j) in enumerate(cells):
        batch[k, :, i:i + size, j:j + size] = blur[0, :, i:i + size, j:j + size]
    scores = predict(batch)
    total, count = torch.zeros(224, 224, device=DEVICE), torch.zeros(224, 224, device=DEVICE)
    for (i, j), s in zip(cells, scores):
        total[i:i + size, j:j + size] += base_score - s
        count[i:i + size, j:j + size] += 1
    return total / count

def curve_auc(heat, mode, steps=100):
    """deletion: из снимка убираем в размытие; insertion: в размытие возвращаем снимок."""
    order = heat.flatten().argsort(descending=True)
    per = len(order) // steps
    cur = (x if mode == 'deletion' else blur).clone()
    src = blur[0] if mode == 'deletion' else x[0]
    curve = [predict(cur).item() / base_score]
    for t in range(steps):
        idx = order[t * per:(t + 1) * per]
        rows, cols = idx // 224, idx % 224
        cur[0, :, rows, cols] = src[:, rows, cols]
        curve.append(predict(cur).item() / base_score)
    return float(np.trapezoid(curve, dx=1 / steps))

torch.manual_seed(0)
heats = {'Meaningful perturbation, 1 - m': 1 - mask_reg,
         'Extremal perturbations, сумма масок': ep_masks.sum(0),
         'Occlusion, окно 64, шаг 16': occlusion(64, 16),
         'случайная карта': torch.rand(224, 224, device=DEVICE)}
print(f'{"карта":38}{"deletion (ниже — лучше)":>26}{"insertion (выше — лучше)":>27}')
for name, heat in heats.items():
    print(f'{name:38}{curve_auc(heat, "deletion"):>26.4f}{curve_auc(heat, "insertion"):>27.4f}')

Обе метрики считаются с размытием `blur` в роли baseline. По deletion впереди occlusion (0.0443 против 0.0619 у Extremal perturbations и 0.0787 у Meaningful perturbation), а по insertion впереди Extremal perturbations: 0.7342 против 0.6952 у Meaningful perturbation и 0.6469 у occlusion. Каждый оптимизационный метод искал маску для своей постановки. Meaningful perturbation искала маску для смеси с тем же `blur`, и по deletion — вопросу игры на удаление — ее карта ближе к occlusion, чем к случайной. Extremal perturbations искала маски для пирамиды размытий `torchray` и только трех площадей, а deletion и insertion проходят все доли площади от 0 до 100%. Карта, оптимальная для одной постановки, не обязана быть лучшей для другой: прежде чем сравнивать методы числом, проверьте, на какой вопрос отвечает метрика и какой порчей она пользуется.

## 4. Real-time saliency: маска за один проход

Каждый запуск в разделах 1 и 2 — сотни прямых и обратных проходов ResNet-50 на одну картинку. Real-time saliency ([Dabkowski и Gal, 2017](https://arxiv.org/abs/1705.07857)) переносит эту оптимизацию во время обучения: отдельная сеть-маскировщик учится выдавать маску по изображению, и при объяснении нужен один ее прямой проход. Целевая функция из урока:

$$L(M)=\lambda_1\,TV(M)+\lambda_2\,AV(M)-\log f_c\bigl(\Phi(X,M)\bigr)+\lambda_3\,f_c\bigl(\Phi(X,1-M)\bigr)^{\lambda_4}.$$

В статье маскировщик обучают на ImageNet поверх ResNet-50, и это часы на GPU. Здесь та же функция на MNIST, где обучение идет минуты даже на CPU:

- объясняемая модель — маленькая сверточная сеть, обученная одну эпоху; дальше ее веса заморожены;
- $\Phi(X,M)=X\odot M$: у MNIST фон нулевой, и ноль — естественная альтернатива $A$;
- маскировщик выдает маску на сетке $7\times7$ и растягивает ее до $28\times28$, это та же защита грубой сеткой, что в разделе 1;
- класс $c$ — тот, который предсказала объясняемая модель, поэтому маскировщику не нужен класс на входе (в статье он есть, потому что маску просят для любого класса);
- $\lambda_1=1,\ \lambda_2=2,\ \lambda_3=5,\ \lambda_4=0.3$.

In [ ]:
mnist_train = datasets.MNIST('data', train=True, download=True)
mnist_test = datasets.MNIST('data', train=False, download=True)
Xtr = mnist_train.data.float().div(255).unsqueeze(1).to(DEVICE)
Xte = mnist_test.data.float().div(255).unsqueeze(1).to(DEVICE)
Ytr, Yte = mnist_train.targets.to(DEVICE), mnist_test.targets.to(DEVICE)

def batches(X, bs=128, shuffle=True):
    idx = torch.randperm(len(X), device=DEVICE) if shuffle else torch.arange(len(X), device=DEVICE)
    for s in range(0, len(X), bs):
        yield idx[s:s + bs]

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 32, 3, padding=1)
        self.c2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc = nn.Linear(64 * 7 * 7, 128)
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.c1(x)), 2)
        x = F.max_pool2d(F.relu(self.c2(x)), 2)
        return self.out(F.relu(self.fc(x.flatten(1))))

torch.manual_seed(0)
clf = SmallCNN().to(DEVICE)
opt = torch.optim.Adam(clf.parameters(), 1e-3)
for j in batches(Xtr):
    loss = F.cross_entropy(clf(Xtr[j]), Ytr[j])
    opt.zero_grad()
    loss.backward()
    opt.step()
clf.eval()
for p in clf.parameters():
    p.requires_grad_(False)
with torch.no_grad():
    pred_te = torch.cat([clf(Xte[j]).argmax(1) for j in batches(Xte, 1000, False)])
print(f'точность объясняемой модели на тесте MNIST: {(pred_te == Yte).float().mean():.4f}')

In [ ]:
class Masker(nn.Module):
    """Изображение 28x28 -> маска 7x7 в (0, 1) -> растяжение до 28x28."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, 1, 1))

    def forward(self, x):
        return F.interpolate(torch.sigmoid(self.net(x)), size=(28, 28), mode='bilinear', align_corners=False)

def tv(m):
    return (((m[..., 1:, :] - m[..., :-1, :]) ** 2).mean((1, 2, 3))
            + ((m[..., :, 1:] - m[..., :, :-1]) ** 2).mean((1, 2, 3)))

def rts_loss(m, xb, c, l1=1.0, l2=2.0, l3=5.0, l4=0.3):
    """L(M) из урока; Phi(X, M) = X * M, альтернатива — нулевой фон MNIST."""
    keep = clf(xb * m).softmax(1).gather(1, c[:, None])[:, 0]
    drop = clf(xb * (1 - m)).softmax(1).gather(1, c[:, None])[:, 0]
    return l1 * tv(m) + l2 * m.mean((1, 2, 3)) - torch.log(keep + 1e-6) + l3 * (drop + 1e-6) ** l4

torch.manual_seed(0)
masker = Masker().to(DEVICE)
opt = torch.optim.Adam(masker.parameters(), 1e-3)
t0 = time.time()
for epoch in range(2):
    for j in batches(Xtr):
        with torch.no_grad():
            c = clf(Xtr[j]).argmax(1)
        loss = rts_loss(masker(Xtr[j]), Xtr[j], c).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
masker.eval()
print(f'обучение маскировщика, 2 эпохи: {time.time() - t0:.0f} с на {DEVICE}')

with torch.no_grad():
    M = masker(Xte)
    kept = (clf(Xte * M).argmax(1) == pred_te).float().mean().item()
    broken = (clf(Xte * (1 - M)).argmax(1) != pred_te).float().mean().item()
print(f'тест MNIST, 10 000 цифр: средняя площадь маски {M.mean():.3f}')
print(f'   класс сохранен под маской: {kept:.3f}; класс сменился, если закрыть маску: {broken:.3f}')

Теперь та же целевая функция, но маска оптимизируется для каждой цифры отдельно, как в разделе 1: параметр — сетка $7\times7$ до сигмоиды, 300 шагов Adam. Сравним на первых 20 цифрах теста время на одно объяснение и значение $L(M)$, которого достигает каждый способ.

In [ ]:
def optimize_one(xb, c, steps=300):
    p = torch.zeros(1, 1, 7, 7, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([p], 0.1)
    for _ in range(steps):
        m = F.interpolate(torch.sigmoid(p), size=(28, 28), mode='bilinear', align_corners=False)
        loss = rts_loss(m, xb, c).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return F.interpolate(torch.sigmoid(p), size=(28, 28), mode='bilinear', align_corners=False).detach()

N = 20
torch.manual_seed(0)
t0 = time.time()
M_opt = torch.cat([optimize_one(Xte[i:i + 1], pred_te[i:i + 1]) for i in range(N)])
t_opt = (time.time() - t0) / N
t0 = time.time()
with torch.no_grad():
    M_rts = masker(Xte[:N])
t_rts = (time.time() - t0) / N
with torch.no_grad():
    loss_opt = rts_loss(M_opt, Xte[:N], pred_te[:N]).mean().item()
    loss_rts = rts_loss(M_rts, Xte[:N], pred_te[:N]).mean().item()
print(f'{"":30}{"время на объяснение":>22}{"L(M), среднее по 20":>22}')
print(f'{"оптимизация по каждой цифре":30}{t_opt * 1e3:>19.1f} мс{loss_opt:>22.3f}')
print(f'{"маскировщик, один проход":30}{t_rts * 1e3:>19.3f} мс{loss_rts:>22.3f}')

Маскировщик отвечает на три порядка быстрее, но его $L(M)$ выше, чем у оптимизации по каждой цифре: одна сеть на все входы не находит для каждой цифры лучшую маску. Этот разрыв называют амортизационным. Вторая оговорка — привязка: маскировщик обучен через градиент конкретной модели `clf` на конкретных данных, и для другой модели его учат заново.

**Задание 3.** Нарисуйте для первых восьми цифр теста три ряда: цифру, маску маскировщика `M_rts` и маску оптимизации `M_opt`. Посчитайте для этих двадцати цифр корреляцию Пирсона двух масок по каждой цифре и выведите ее медиану. Во сколько раз маскировщик быстрее оптимизации на вашем устройстве?

In [ ]:
# Ваш код здесь

## 5. L2X: выбрать $k$ блоков

L2X ([Chen, Song, Wainwright и Jordan, 2018](https://arxiv.org/abs/1802.07814)) тоже амортизирует, но вместо гладкой маски выбирает ровно $k$ признаков. Признаки здесь — блоки $4\times4$ пикселя, у цифры $28\times28$ их 49. Обучаются две сети:

- объяснитель $w_\theta(X)$ выдает 49 логитов; из них берут $k$ независимых Concrete-величин при температуре $\tau=0.1$, как в статье, и складывают покоординатным максимумом, $V_i=\max_j C^j_i$, — получается мягкий $k$-hot вектор, по которому идет градиент;
- вспомогательная модель $q$ (та же архитектура `SmallCNN`) по входу $X\odot V$ предсказывает ответ объясняемой модели. Ее кросс-энтропия — вариационная нижняя оценка взаимной информации $I(X_S;Y)$ из урока. Обе сети учит RMSprop с шагом 0.001, как в статье.

На тесте выбор жесткий: $k$ блоков с наибольшими логитами. Качество выбора меряют post-hoc точностью: доля цифр, на которых модель по одним выбранным блокам (остальное — нулевой фон) дает тот же ответ, что по всей цифре. Посчитаем ее дважды — для объясняемой модели `clf` и для вспомогательной $q$, — и для сравнения у $k$ случайных блоков.

In [ ]:
PATCH, SIDE = 4, 7          # блоки 4x4, сетка 7x7, 49 признаков

class Explainer(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(64, 1, 1))

    def forward(self, x):
        return self.net(x).flatten(1)          # (B, 49) — логиты w_theta(X)

def to_pixels(v):
    return v.view(-1, 1, SIDE, SIDE).repeat_interleave(PATCH, 2).repeat_interleave(PATCH, 3)

def concrete_khot(logits, k, tau):
    """k независимых Concrete-величин и покоординатный максимум: мягкий k-hot вектор V."""
    u = torch.rand(logits.shape[0], k, logits.shape[1], device=DEVICE).clamp_min(1e-10)
    gumbel = -torch.log(-torch.log(u))
    return ((logits[:, None, :] + gumbel) / tau).softmax(-1).amax(1)

def hard_khot(logits, k):
    return torch.zeros_like(logits).scatter(1, logits.topk(k, 1).indices, 1.0)

def train_l2x(k, tau=0.1, epochs=1, seed=0):
    torch.manual_seed(seed)
    explainer, approx = Explainer().to(DEVICE), SmallCNN().to(DEVICE)
    opt = torch.optim.RMSprop(list(explainer.parameters()) + list(approx.parameters()), 1e-3)   # как в статье
    for epoch in range(epochs):
        for j in batches(Xtr):
            with torch.no_grad():
                c = clf(Xtr[j]).argmax(1)
            v = concrete_khot(explainer(Xtr[j]), k, tau)
            loss = F.cross_entropy(approx(Xtr[j] * to_pixels(v)), c)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return explainer.eval(), approx.eval()

def evaluate_l2x(explainer, approx, k):
    with torch.no_grad():
        chosen = Xte * to_pixels(hard_khot(explainer(Xte), k))
        random_k = Xte * to_pixels(hard_khot(torch.rand(len(Xte), SIDE * SIDE, device=DEVICE), k))
        return {'clf, выбранные блоки': (clf(chosen).argmax(1) == pred_te).float().mean().item(),
                'q, выбранные блоки': (approx(chosen).argmax(1) == pred_te).float().mean().item(),
                'clf, случайные блоки': (clf(random_k).argmax(1) == pred_te).float().mean().item()}

K = 4
t0 = time.time()
explainer4, approx4 = train_l2x(K)
print(f'обучение L2X, k = {K}, одна эпоха: {time.time() - t0:.0f} с на {DEVICE}')
for name, acc in evaluate_l2x(explainer4, approx4, K).items():
    print(f'   post-hoc точность, {name:22} {acc:.3f}')

In [ ]:
with torch.no_grad():
    sel = to_pixels(hard_khot(explainer4(Xte[:8]), K))
fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for i in range(8):
    axes[0, i].imshow(Xte[i, 0].cpu().numpy(), cmap='gray')
    axes[1, i].imshow((Xte[i, 0] * (0.25 + 0.75 * sel[i, 0])).cpu().numpy(), cmap='gray')
    axes[1, i].set_title(f'ответ clf: {pred_te[i].item()}', fontsize=9)
    for row in (0, 1):
        axes[row, i].axis('off')
plt.suptitle(f'L2X: {K} выбранных блока 4x4 яркие, остальное притушено', fontsize=10)
plt.tight_layout()
plt.show()

Выбранные блоки сохраняют ответ объясняемой модели чаще, чем столько же случайных (первая строка против третьей), — объяснитель выучил, где у цифры решающие штрихи. Но вспомогательная модель $q$ по тем же блокам угадывает ответ `clf` чаще, чем сама `clf` (вторая строка против первой). Объяснитель обучался вместе с $q$ и выбирал блоки, понятные именно ей: $q$ видела только такие обрезанные входы и научилась их читать, а `clf` училась на целых цифрах. Поэтому выбранное подмножество описывает связку «объяснитель и $q$», и чем сильнее $q$ отличается от объясняемой модели, тем меньше оно говорит о `clf`. Это ограничение из карточки метода, и post-hoc точность по `clf` — способ его измерить.

**Задание 4.** Обучите L2X при $k=2$ и $k=8$ (функция `train_l2x`) и сведите в таблицу три post-hoc точности для $k=2,4,8$. Как с ростом $k$ меняется разрыв между точностью по $q$ и по `clf`? Какое $k$ вы бы выбрали, если объяснение должно сохранять ответ самой `clf` хотя бы на 80% цифр?

In [ ]:
# Ваш код здесь

## Что унести из тетради

- **Маска — обучаемый параметр, метод — короткий цикл оптимизации.** Meaningful perturbation целиком — функция `deletion_game`: маска, смесь с baseline, штраф за площадь, Adam.
- **Регуляризация — часть определения метода.** Маска в полном разрешении без штрафов решает задачу оптимизации состязательным узором; грубая сетка, полная вариация и дрожание делают из нее объяснение.
- **$\lambda$ или площадь.** Meaningful perturbation дает семейство масок по $\lambda$, Extremal perturbations — по площади, заданной в понятных единицах. Игры на удаление и на сохранение не обязаны давать одну и ту же область.
- **Метрика зависит от своей порчи.** По deletion с размытием впереди occlusion, по insertion впереди Extremal perturbations, а occlusion позади обеих оптимизационных карт: маски Meaningful perturbation и Extremal perturbations оптимальны для своей порчи и своих площадей, а не для порядка пикселей, по которому идет метрика.
- **Амортизация меняет время на объяснение на обучение.** Маскировщик Real-time saliency отвечает за один проход, но уступает оптимизации по каждому входу в значении целевой функции и привязан к своей модели.
- **L2X описывает свою вспомогательную модель.** Post-hoc точность по объясняемой модели ниже, чем по вспомогательной: проверять выбранное подмножество надо на той модели, которую объясняют.

**Метки:** локальные, post-hoc. Meaningful perturbation, Extremal perturbations и Real-time saliency — model-specific, нужен градиент по маске; L2X — model-agnostic, нужны только ответы модели. Выход: маска размера входа (или $k$ выбранных признаков у L2X).